# Distributed Spam Detection

For presentational purposes I assume Ray is running on port 8080.

In [1]:
import ray
from ray.air.integrations.keras import ReportCheckpointCallback

In [2]:
if ray.is_initialized:
    ray.shutdown()

ray.init()

2026-05-31 14:51:14,337	INFO worker.py:2012 -- Started a local Ray instance.
C:\Users\krzys\Documents\Programy_Python\DistributedMLProject\.venv\Lib\site-packages\ray\_private\worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Python version:,3.12.12
Ray version:,2.55.1



## Data loading and preprocessing

In [3]:
import pandas as pd

local_path = r"data\enron_spam_data.csv"

pdf = pd.read_csv(
    local_path,
    on_bad_lines='skip',
    encoding='utf-8',
    low_memory=False
)

pdf = pdf.fillna({"Message": "", "Subject": ""})

df = ray.data.from_pandas(pdf)
df.show(5)

2026-05-31 14:51:26,003	INFO dataset.py:3818 -- Tip: Use `take_batch()` instead of `take() / show()` to return records in pandas or numpy batch format.
2026-05-31 14:51:26,029	INFO logging.py:416 -- Registered dataset logger for dataset dataset_1_0
2026-05-31 14:51:26,072	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_1_0. Full logs are in C:\Users\krzys\AppData\Local\Temp\ray\session_2026-05-31_14-51-10_009137_32532\logs\ray-data
2026-05-31 14:51:26,073	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_1_0: InputDataBuffer[Input] -> LimitOperator[limit=5]
2026-05-31 14:51:26,075	WARNING resource_manager.py:169 -- ⚠️  Ray's object store is configured to use only 42.9% of available memory (1.0GiB out of 2.4GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJE

{'Message ID': 0, 'Subject': 'christmas tree farm pictures', 'Message': '', 'Spam/Ham': 'ham', 'Date': '1999-12-10'}
{'Message ID': 1, 'Subject': 'vastar resources , inc .', 'Message': 'gary , production from the high island larger block a - 1 # 2 commenced on\nsaturday at 2 : 00 p . m . at about 6 , 500 gross . carlos expects between 9 , 500 and\n10 , 000 gross for tomorrow . vastar owns 68 % of the gross production .\ngeorge x 3 - 6992\n- - - - - - - - - - - - - - - - - - - - - - forwarded by george weissman / hou / ect on 12 / 13 / 99 10 : 16\nam - - - - - - - - - - - - - - - - - - - - - - - - - - -\ndaren j farmer\n12 / 10 / 99 10 : 38 am\nto : carlos j rodriguez / hou / ect @ ect\ncc : george weissman / hou / ect @ ect , melissa graves / hou / ect @ ect\nsubject : vastar resources , inc .\ncarlos ,\nplease call linda and get everything set up .\ni \' m going to estimate 4 , 500 coming up tomorrow , with a 2 , 000 increase each\nfollowing day based on my conversations with bill fis

In [4]:
def preprocess_batch(batch):
    subject_clean = batch["Subject"].fillna("")
    message_clean = batch["Message"].fillna("")

    batch["Text"] = subject_clean + " " + message_clean

    batch["Spam/Ham"] = batch["Spam/Ham"].map({"ham": 0, "spam": 1})

    return batch

processed_df = df.map_batches(preprocess_batch, batch_format="pandas")

train, test = processed_df.train_test_split(test_size=0.2, shuffle=True)
print("Train set:")
train.show(1)
print("Test set:")
test.show(1)

2026-05-31 14:51:26,913	INFO logging.py:416 -- Registered dataset logger for dataset dataset_4_0
2026-05-31 14:51:26,919	INFO streaming_executor.py:166 -- Starting execution of Dataset dataset_4_0. Full logs are in C:\Users\krzys\AppData\Local\Temp\ray\session_2026-05-31_14-51-10_009137_32532\logs\ray-data
2026-05-31 14:51:26,920	INFO streaming_executor.py:167 -- Execution plan of Dataset dataset_4_0: InputDataBuffer[Input] -> AllToAllOperator[MapBatches(preprocess_batch)->RandomShuffle]
2026-05-31 14:51:26,927	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_4_0 =======
2026-05-31 14:51:26,929	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-05-31 14:51:26,929	INFO logging_progress.py:227 -- Active & requested resources: 0/12 CPU, 0.0B/527.7MiB object store
2026-05-31 14:51:26,930	INFO logging_progress.py:181 -- 
2026-05-31 14:51:26,930	INFO logging_progress.py:231 -- MapBatches(preprocess_batch)->RandomShuffle: 0/1
2026-05-31 14:51:26,930	INFO logging_progress

Train set:
{'Message ID': 10171, 'Subject': 're : [ 3 ]', 'Message': 'this will be our closing effort\nwe have aimed to speak to you on multiple possibilities and we await your response now !\nyour exisiting loan situation certifies you for up to a 3 . 60 % lower rate .\nhowever , since our previous attempts to speak to\nyou have failed , this will be our last notice to close for you the lower rate .\nplease end this final step upon receiving\nthis notice immediately , and complete your request for information now .\napply here .\nif your decision is not to make use of this final offer going here will help you to do so .', 'Spam/Ham': 1, 'Date': '2005-06-25', 'Text': 're : [ 3 ] this will be our closing effort\nwe have aimed to speak to you on multiple possibilities and we await your response now !\nyour exisiting loan situation certifies you for up to a 3 . 60 % lower rate .\nhowever , since our previous attempts to speak to\nyou have failed , this will be our last notice to close for

## Training a model

In [5]:
import tensorflow as tf

def build_model() -> tf.keras.Sequential:
    model = tf.keras.Sequential([
        tf.keras.layers.InputLayer(input_shape=()),
        tf.keras.layers.TextVectorization(max_tokens=10000, output_sequence_length=200),
        tf.keras.layers.Embedding(input_dim=10000, output_dim=32),
        tf.keras.layers.GlobalAveragePooling1D(),
        tf.keras.layers.Dense(100, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])
    return model

In [6]:
from ray.train.tensorflow.keras import ReportCheckpointCallback

def training_loop(config: dict):
    batch_size = config.get("batch_size", 32)
    epochs = config.get("epochs", 4)

    strategy = tf.distribute.MultiWorkerMirroredStrategy()
    with strategy.scope():
        model = build_model()
        model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

    data = ray.train.get_dataset_shard("train")

    tf_dataset = data.to_tf(feature_columns=["Text"], label_columns="Spam/Ham", batch_size=batch_size)

    model.fit(tf_dataset,
              callbacks=[ReportCheckpointCallback(metrics=["accuracy"])],
              epochs=epochs)


SyntaxError: invalid syntax (2366794020.py, line 10)